In [1]:
"""
stage0_events_i2.py -- Stage 0 (iteration 2): event extraction + run manifest.

Changes vs i1:
  - SESSION window [SESSION_START, SESSION_END) applied in load(), before
    bar_index is assigned (bar_index stays positional-from-first-kept-bar).
  - Writes a small run-manifest sidecar (stage0_manifest_{FRAME}s.json) recording
    the fork axes (frame, session window, stream set, warmup, input file). This is
    the CONTRACT that downstream stages load -- kept separate from the diagnostics
    report (different lifetime: manifest is loaded, diagnostics is read once).
  - Everything else (signfill/pivots/diagnostics, naive timestamps, `date`,
    WARMUP in bars, extremum_timestamp) is unchanged from i1.

RULES (unchanged from i1 unless noted)
  S0.1  Sign of a series is sign(diff), zero-runs carried forward from last
        non-zero sign; leading run before first non-zero sign is 0 (masked).
  S0.2  Pivot of x at bar k is emitted as an EVENT at bar k+1, first knowable bar.
  S0.3  Target (MNQ JMA pivot) obeys S0.2: target at t <=> leg_dir[t] != leg_dir[t-1].
  S0.5  Polarity: +1 = local maximum at extremum_bar, -1 = local minimum.
  S0.6  opposing = polarity * leg_dir at event bar; +1 opposing, -1 confirming, 0 undef.
  S0.7  Per-session (no cross-day spillage); first WARMUP_BARS bars masked.
  S0.8  One bar grid, one file, co-knowable at bar close.
  S0.9  Diagnostics report the per-stream recall ceiling.
  S0.10 d1 centerline auto-detected for the sign-agreement diagnostic only.
  S0.11 (new) SESSION window is applied here and recorded in the manifest; bar_index
        is positional over the FILTERED, timestamp-sorted frame. Naive timestamps,
        no tz handling -- data prep is responsible for timestamp correctness.
"""

import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- config (fork axes)
COLS = {
    "timestamp": "timestamp",
    "date": "date",
    "jma": "JMA",
    "d1": "jmaD1",
    "d2": "jmaD2",
    "t_d1": "tickJmaD1",
    "t_d2": "tickJmaD2",
}

# logical stream name -> COLS key. The single origin of the stream set; downstream
# stages read this back from the manifest rather than redeclaring it.
STREAMS = [("MNQ_D1", "d1"), ("MNQ_D2", "d2"), ("TICK_D1", "t_d1"), ("TICK_D2", "t_d2")]

FRAME = 3
SESSION_START = "09:30"          # inclusive
SESSION_END = "16:00"            # exclusive
WARMUP_BARS = 10                 # bars, per-session (resolution-natural, stays in bars)
LAG_SUPPORTS = [1, 2, 3, 5, 8, 13, 21, 34]

SOURCE_FILE = f"../data/mnq-tick-oscillator-{FRAME}sec.pqt"   # Stage 0 input
RAW_FILE    = f"../data/mnq-ohlc-raw-{FRAME}sec.pqt"             # pure MNQ OHLC; Stage 5 pairs it
OUT_DIR     = "stage-0"

In [2]:
# ---------------------------------------------------------------- primitives
def signfill(x):
    """S0.1"""
    d = np.zeros(len(x), dtype=np.float64)
    d[1:] = np.diff(x)
    s = np.sign(d).astype(np.int8)
    pos = np.maximum.accumulate(np.where(s != 0, np.arange(len(s)), -1))
    out = np.zeros(len(s), dtype=np.int8)
    ok = pos >= 0
    out[ok] = s[pos[ok]]
    return out


def pivots(s):
    """S0.2, S0.5. s is a signfill array. Returns (event_bar, extremum_bar, polarity)."""
    if len(s) < 2:
        e = np.empty(0, dtype=np.int64)
        return e, e, np.empty(0, dtype=np.int8)
    turn = (s[1:] != s[:-1]) & (s[1:] != 0) & (s[:-1] != 0)
    event_bar = np.nonzero(turn)[0] + 1
    return event_bar, event_bar - 1, s[event_bar - 1]

In [3]:
# ---------------------------------------------------------------- load
def load(data_path, session_start, session_end):
    df = pd.read_parquet(data_path)
    df = df.rename(columns={v: k for k, v in COLS.items()})
    df = df[list(COLS.keys())].copy()

    t = df["timestamp"].dt.time                                       # S0.11
    lo = pd.Timestamp(session_start).time()
    hi = pd.Timestamp(session_end).time()
    n_pre = len(df)
    df = df[(t >= lo) & (t < hi)]
    n_session = len(df)

    df = df.sort_values("timestamp").reset_index(drop=True)
    n1 = len(df)
    df = df.dropna(subset=["jma", "d1", "d2", "t_d1", "t_d2"]).reset_index(drop=True)
    df["bar_index"] = np.arange(len(df), dtype=np.int64)             # positional, post-filter
    return df, {"rows_in_file": n_pre, "rows_in_session": n_session,
                "rows_dropped_nan": n1 - len(df)}

In [4]:
# ---------------------------------------------------------------- extraction
def extract(df):
    ev_rows = []
    bar_frames = []

    for sess, g in df.groupby("date", sort=True):
        g = g.reset_index(drop=True)
        if len(g) <= WARMUP_BARS + 2:
            continue
        gbar = g["bar_index"].to_numpy()

        jma = g["jma"].to_numpy(np.float64)
        leg_dir = signfill(jma)                                     # S0.1
        t_ev, t_ex, t_pol = pivots(signfill(jma))                   # S0.3 (target)

        keep = t_ev >= WARMUP_BARS                                  # S0.7
        t_ev, t_ex, t_pol = t_ev[keep], t_ex[keep], t_pol[keep]

        is_target = np.zeros(len(g), dtype=bool)
        is_target[t_ev] = True
        leg_id = np.cumsum(is_target)
        starts = np.concatenate(([0], t_ev))
        leg_start = starts[leg_id]
        leg_age = np.arange(len(g)) - leg_start
        leg_amp = np.abs(jma - jma[leg_start])

        bar_frames.append(pd.DataFrame({
            "bar_index": gbar,
            "timestamp": g["timestamp"].to_numpy(),
            "date": sess,
            "jma": jma,
            "d1": g["d1"].to_numpy(np.float64),
            "jma_leg_dir": leg_dir,
            "leg_id": leg_id.astype(np.int32),
            "leg_age": leg_age.astype(np.int32),
            "leg_amp": leg_amp,
            "is_target": is_target,
            "warm": np.arange(len(g)) < WARMUP_BARS,
        }))

        for name, col in STREAMS + [("MNQ_JMA_SELF", "jma")]:
            x = g[col].to_numpy(np.float64)
            e, k, pol = pivots(signfill(x))
            m = e >= WARMUP_BARS                                    # S0.7
            e, k, pol = e[m], k[m], pol[m]
            if len(e) == 0:
                continue
            xk = x[k]
            prev_e = np.concatenate(([-1], e[:-1]))
            prev_xk = np.concatenate(([np.nan], xk[:-1]))
            ev_rows.append(pd.DataFrame({
                "date": sess,
                "stream": name,
                "event_bar": gbar[e],
                "extremum_bar": gbar[k],
                "timestamp": g["timestamp"].to_numpy()[e],
                "extremum_timestamp": g["timestamp"].to_numpy()[k],
                "polarity": pol,                                    # S0.5
                "jma_leg_dir": leg_dir[e],                          # S0.6
                "opposing": (pol * leg_dir[e]).astype(np.int8),     # S0.6
                "value": xk,
                "prev_leg_amp": np.abs(xk - prev_xk),
                "bars_since_prev": np.where(prev_e < 0, -1, e - prev_e).astype(np.int32),
                "leg_age_at_event": leg_age[e].astype(np.int32),
            }))

    bars = pd.concat(bar_frames, ignore_index=True)
    events = pd.concat(ev_rows, ignore_index=True).sort_values(
        ["event_bar", "stream"]).reset_index(drop=True)
    events["stream_id"] = events["stream"].astype("category").cat.codes.astype(np.int8)
    return bars, events

In [5]:
# ---------------------------------------------------------------- diagnostics
def diagnostics(bars, events, load_counts):
    tgt = np.sort(bars.loc[bars["is_target"], "bar_index"].to_numpy())
    n_bars = int((~bars["warm"]).sum())

    d1 = bars["d1"].to_numpy()
    center = 100.0 if (np.nanmin(d1) > 0 and 90.0 < np.nanmedian(d1) < 110.0) else 0.0   # S0.10
    agree = float((np.sign(d1 - center) == bars["jma_leg_dir"].to_numpy()).mean())

    d = {
        "n_bars": int(len(bars)),
        "n_bars_ex_warmup": n_bars,
        "n_sessions": int(bars["date"].nunique()),
        "rows_in_file": load_counts["rows_in_file"],
        "rows_in_session": load_counts["rows_in_session"],
        "rows_dropped_nan": load_counts["rows_dropped_nan"],
        "n_target": int(len(tgt)),
        "target_rate_per_1k": round(1000.0 * len(tgt) / max(n_bars, 1), 3),
        "median_leg_bars": float(np.median(np.diff(tgt))) if len(tgt) > 2 else None,
        "d1_center_detected": center,
        "d1_sign_agrees_leg_dir": agree,
        "streams": {},
        "recall_ceiling": {},
        "precision_base": {},
    }

    for name, g in events.groupby("stream"):
        eb = np.sort(g["event_bar"].to_numpy())
        d["streams"][name] = {
            "n": int(len(g)),
            "rate_per_1k": round(1000.0 * len(g) / max(n_bars, 1), 3),
            "ratio_to_target": round(len(g) / max(len(tgt), 1), 2),
            "median_inter_event_bars": float(np.median(np.diff(eb))) if len(eb) > 2 else None,
            "frac_opposing": float((g["opposing"] == 1).mean()),
        }

    opp = events[events["opposing"] == 1]
    for name, g in opp.groupby("stream"):                           # S0.9
        eb = np.sort(g["event_bar"].to_numpy())
        rec, prec = {}, {}
        for L in LAG_SUPPORTS:
            lo = np.searchsorted(eb, tgt - L, side="left")
            hi = np.searchsorted(eb, tgt, side="left")
            rec[L] = round(float((hi > lo).mean()), 4)
            a = np.searchsorted(tgt, eb + 1, side="left")
            b = np.searchsorted(tgt, eb + L, side="right")
            prec[L] = round(float((b > a).mean()), 4)
        d["recall_ceiling"][name] = rec
        d["precision_base"][name] = prec

    eb = np.sort(opp["event_bar"].unique())
    d["recall_ceiling"]["ANY_OPPOSING"] = {
        L: round(float((np.searchsorted(eb, tgt, "left")
                        > np.searchsorted(eb, tgt - L, "left")).mean()), 4)
        for L in LAG_SUPPORTS
    }
    return d

In [6]:
# ---------------------------------------------------------------- manifest
def build_manifest(bars):
    """The CONTRACT downstream stages load. Fork axes only -- small and stable."""
    return {
        "frame_seconds": FRAME,
        "session_start": SESSION_START,
        "session_end": SESSION_END,
        "warmup_bars": WARMUP_BARS,
        "streams": [{"name": n, "column": COLS[c]} for n, c in STREAMS],
        "self_stream": "MNQ_JMA_SELF",
        "source_file": SOURCE_FILE,
        "raw_file": RAW_FILE,        
        "n_bars": int(len(bars)),
        "n_sessions": int(bars["date"].nunique()),
        "date_min": str(bars["date"].min()),
        "date_max": str(bars["date"].max()),
        "created": pd.Timestamp.now().isoformat(),
    }

In [7]:
# ---------------------------------------------------------------- main
df, load_counts = load(SOURCE_FILE, SESSION_START, SESSION_END)

print(df.info())
print(df.head())

bars, events = extract(df)
diag = diagnostics(bars, events, load_counts)
manifest = build_manifest(bars)

bars.to_parquet(f"{OUT_DIR}/bars_{FRAME}s.pqt", index=False)
events.to_parquet(f"{OUT_DIR}/events_{FRAME}s.pqt", index=False)

with open(f"{OUT_DIR}/stage0_diagnostics_{FRAME}s.json", "w") as f:
    json.dump(diag, f, indent=2, default=str)

with open(f"{OUT_DIR}/stage0_manifest_{FRAME}s.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(json.dumps(manifest, indent=2, default=str))
print(json.dumps(diag, indent=2, default=str))

FileNotFoundError: [Errno 2] No such file or directory: '../data/mnq-tick-oscillator-3sec.pqt'